# Otimização — capítulo 3: Otimização multidimensional sem restrições

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_3_conjugate_gradients/ex03_3_3_conjugate_gradients.ipynb)

Corra a célula **Preparação** e depois o exemplo (no Colab: *Runtime → Run all*). No fim, o exemplo compara os resultados com os slides (*confere com os slides: sim*). Os slides usam vírgula decimal; aqui usa-se o ponto. Caderno gerado a partir de `ex03_3_3_conjugate_gradients.py` por `notebooks/make_notebooks.py`.

*Code comments and printed messages are in Portuguese.*

## Preparação

In [1]:
# Preparação: põe as funções da UC no caminho do Python.
# No Colab (ou noutra pasta), clona o repositório; dentro do repositório, usa code/python.
import os, sys
LOCAL = [p for p in ("..", os.path.join("..", "..")) if os.path.isfile(os.path.join(p, "uc_setup.py"))]
if LOCAL:
    RAIZ = os.path.abspath(LOCAL[0])
else:
    if not os.path.isdir("optimization-course"):
        !git clone -q --depth 1 https://github.com/aguilarmadeira/optimization-course
    RAIZ = os.path.abspath(os.path.join("optimization-course", "code", "python"))
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import uc_setup  # acrescenta as pastas dos métodos e common/
print("Código da UC pronto.")

Código da UC pronto.


## 3.3.3 Método dos gradientes conjugados (Fletcher–Reeves)

Exemplo [`ex03_3_3_conjugate_gradients.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_3_conjugate_gradients/ex03_3_3_conjugate_gradients.py) · funções da UC: [`conj_grad.py`](https://github.com/aguilarmadeira/optimization-course/blob/main/code/python/03_multivariate_optimization/03_3_3_conjugate_gradients/conj_grad.py)

```text
Reproduz os exemplos do deck 3.3.3 (gradientes conjugados, Fletcher-Reeves).

Quadrática f(x) = (x1^2 + 9 x2^2)/2 de x0 = (9, 1), pesquisa em linha exata:
    o 1.º passo é o do gradiente, x1 = (7.2; -0.8); beta_1 = 103.68/162 = 0.64,
    d1 = (-12.96; 1.44), d0^T A d1 = 0 (A-conjugadas), alpha_1 = 5/9,
    x2 = (0, 0): 2 iterações, n_g = 3.
Rosenbrock de (-1.5; 2), ||g|| < 1e-4: com reinício a cada n = 2, 36 it.
    (n_g = 37); sem reinício, 102 it.
«No computador»: com uma pesquisa do tipo fminbnd de tolerância por
    omissão, FR no Rosenbrock pode não convergir (informativo).

Imprime as tabelas, os valores finais e as contagens, e no fim compara com
os slides (vírgula decimal nos slides, ponto aqui).
```

In [2]:
import numpy as np

from conj_grad import conj_grad


def confere(v, s, d):
    """v confere com o valor s do slide, mostrado com d casas decimais?"""
    v = np.ravel(np.asarray(v, float)); s = np.ravel(np.asarray(s, float))
    d = np.ravel(np.asarray(d, float))
    return bool(np.all(np.abs(v - s) <= 0.5 * 10.0 ** (-d) * (1 + 1e-6)))


def simnao(c):
    return tuple("sim" if ci else "não" for ci in c)


ok = True
print("Otimização — deck 3.3.3: gradientes conjugados (Fletcher-Reeves)")

# ------------------------------------------------ quadrática
A = np.diag([1.0, 9.0])
quad = lambda x: 0.5 * (x[0]**2 + 9 * x[1]**2)     # = x^T A x / 2
gq = lambda x: np.array([x[0], 9 * x[1]])
x0 = np.array([9.0, 1.0])
print("\nQuadrática (x1^2 + 9 x2^2)/2 de (9, 1), tolg = 1e-6, pesquisa em linha de Brent (tol 1e-10):")
r = conj_grad(quad, gq, x0, 1e-6, 100, verbose=True)
H = r.history                          # [k, x1, x2, f, ||g||, alpha, beta, d1, d2]
g0, g1 = gq(H[0, 1:3]), gq(H[1, 1:3])
d0, d1 = H[0, 7:9], H[1, 7:9]
print("||g1||^2/||g0||^2 = %.2f/%.0f = %.4f; d0^T A d1 = %.1e; alpha_1 = %.6f (5/9 = %.6f)"
      % (g1 @ g1, g0 @ g0, H[1, 6], d0 @ A @ d1, H[2, 5], 5 / 9))
print("%s; n_g = %d, n_f = %d" % (r.message, r.ngev, r.nfev))
c = [confere(H[1, 1:3], [7.2, -0.8], 4), confere(g1 @ g1, 103.68, 2), confere(H[1, 6], 0.64, 4),
     confere(d1, [-12.96, 1.44], 2), abs(d0 @ A @ d1) < 1e-4, confere(H[2, 5], 5 / 9, 6),
     confere(r.x, [0, 0], 6), r.nit == 2, r.ngev == 3]
print("  x1: %s | ||g1||^2 = 103.68: %s | beta_1 = 0.64: %s | d1: %s | d0'Ad1 = 0: %s | alpha_1 = 5/9: %s"
      " | x2 = (0, 0): %s | 2 it.: %s | n_g = 3: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------------ Rosenbrock
rosen = lambda x: (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2
grosen = lambda x: np.array([-2 * (1 - x[0]) - 400 * x[0] * (x[1] - x[0]**2), 200 * (x[1] - x[0]**2)])
xr = np.array([-1.5, 2.0])
print("\nRosenbrock de (-1.5; 2), tolg = 1e-4:")
rc = conj_grad(rosen, grosen, xr, 1e-4, 500)                # reinício a cada n = 2
rs = conj_grad(rosen, grosen, xr, 1e-4, 500, restart=0)     # sem reinício periódico
print("FR com reinício a cada n = 2: %d it., n_g = %d, n_f = %d, f = %.1e (%d reinícios)"
      % (rc.nit, rc.ngev, rc.nfev, rc.fx, rc.nrestart))
print("FR sem reinício periódico:    %d it., n_g = %d, n_f = %d, f = %.1e (%d reinícios por g^T d >= 0)"
      % (rs.nit, rs.ngev, rs.nfev, rs.fx, rs.nrestart))
c = [rc.nit == 36, rc.ngev == 37, rs.nit == 102, rc.flag == 0 and rs.flag == 0]
print("  36 it. com reinício: %s | n_g = 37: %s | 102 it. sem: %s | convergem: %s" % simnao(c))
ok = ok and all(c)

# ------------------------------------------- pesquisa tipo fminbnd (informativo)
rb = conj_grad(rosen, grosen, xr, 1e-4, 500, ls="fminbnd")
print("\nCom a pesquisa do tipo fminbnd (TolX = 1e-4) e reinício a cada n: %s; f = %.2g"
      % (rb.message, rb.fx))
print("  (o slide diz que «pode não convergir»: %s)" % ("não convergiu" if rb.flag else "convergiu"))

print("\nconfere com os slides: %s" % ("sim" if ok else "não"))


Otimização — deck 3.3.3: gradientes conjugados (Fletcher-Reeves)

Quadrática (x1^2 + 9 x2^2)/2 de (9, 1), tolg = 1e-6, pesquisa em linha de Brent (tol 1e-10):
   k                    x_k         f(x_k)     ||g_k||     alpha     beta                    d_k
   0  (   9.0000;    1.0000)      45.000000    1.27e+01       nan      nan  (  -9.0000;   -9.0000)
   1  (   7.2000;   -0.8000)      28.800000    1.02e+01    0.2000   0.6400  ( -12.9600;    1.4400)
   2  (  -0.0000;    0.0000)       0.000000    7.36e-13    0.5556      nan  (      nan;       nan)
||g1||^2/||g0||^2 = 103.68/162 = 0.6400; d0^T A d1 = -7.9e-07; alpha_1 = 0.555556 (5/9 = 0.555556)
||g|| <= tolg em 2 iterações; n_g = 3, n_f = 47
  x1: sim | ||g1||^2 = 103.68: sim | beta_1 = 0.64: sim | d1: sim | d0'Ad1 = 0: sim | alpha_1 = 5/9: sim | x2 = (0, 0): sim | 2 it.: sim | n_g = 3: sim

Rosenbrock de (-1.5; 2), tolg = 1e-4:
FR com reinício a cada n = 2: 36 it., n_g = 37, n_f = 775, f = 1.5e-12 (17 reinícios)
FR sem reinício periódi